# Day 14 - Append Overwrite and Partitioning

**Topic:** Append Overwrite and Partitioning  
**Main environment:** Microsoft Fabric Notebook + Fabric Lakehouse  
**Focus:** เข้าใจ write modes เช่น `append`, `overwrite` และการใช้ `partitionBy` เพื่อจัด file layout ของ Lakehouse table อย่างปลอดภัย

วันนี้เราจะฝึกเขียนข้อมูลลง Lakehouse table ด้วย PySpark โดยเน้น mindset ที่สำคัญมากในงาน Data Engineering: ถ้าใช้ `append` ผิดอาจเกิดข้อมูลซ้ำ ถ้าใช้ `overwrite` ผิดอาจลบข้อมูลเดิมทั้ง table และถ้าเลือก partition column ไม่ดีอาจทำให้ table ดูแลยากหรือ query ช้าลง

## Goal of the day

1. แยกให้ออกว่าเมื่อไรควรใช้ **append** และเมื่อไรควรใช้ **overwrite**
2. อธิบายได้ว่า `overwrite` เป็น operation ที่ต้องระวัง เพราะอาจแทนข้อมูลเดิมทั้ง table
3. ใช้ `partitionBy` ตอน write Lakehouse table ได้
4. เข้าใจว่า partition column มีผลต่อ file layout และ query/filter pattern
5. เริ่มระวังปัญหา duplicate rows, accidental overwrite, over-partitioning และ small files

## Why it matters in production

ใน production pipeline เรื่อง write mode และ partitioning สำคัญมาก เพราะ:

- `append` เหมาะกับการเพิ่ม batch ใหม่ แต่ถ้า rerun แบบไม่ออกแบบ idempotency อาจเกิด duplicate records
- `overwrite` เหมาะกับการสร้าง snapshot หรือ test table ใหม่ แต่ถ้าใช้ผิด table อาจลบ history หรือข้อมูล production ได้
- `partitionBy` ช่วยให้ query ที่ filter ตาม partition column มีโอกาสลดปริมาณข้อมูลที่ต้องอ่าน เพราะ Spark สามารถอ่านเฉพาะ partition ที่เกี่ยวข้องได้ ซึ่งอาจทำให้ query เร็วขึ้น
- partition column ที่ cardinality (จำนวนค่าที่แตกต่างกัน) สูงเกินไป เช่น `transaction_id` หรือ `customer_id` อาจสร้าง folder/files จำนวนมากเกินจำเป็น
- การเข้าใจ file layout เป็นพื้นฐานก่อนเรียนเรื่อง MERGE, incremental load, idempotency, OPTIMIZE และ small files ในวันถัด ๆ ไป

Production takeaway:

> Write mode คือ decision ที่กระทบความถูกต้องของข้อมูล ส่วน partitioning คือ decision ที่กระทบทั้ง performance และ maintenance ของ Lakehouse table

## Key concepts

| Concept | Meaning |
|---|---|
| `append` | เพิ่ม rows ใหม่เข้า table เดิม โดยไม่ลบข้อมูลเก่า |
| `overwrite` | เขียนข้อมูลใหม่แทนข้อมูลเดิมของ table หรือ path นั้น |
| `overwriteSchema` | อนุญาตให้ overwrite แล้วเปลี่ยน schema ของ table ได้ ควรใช้แบบตั้งใจเท่านั้น |
| `partitionBy` | กำหนด column ที่ใช้แบ่ง file layout ตอน write เช่น year/month/date |
| Partition column | Column ที่ใช้จัดกลุ่ม files/folders ของ table เพื่อช่วยการอ่านบาง pattern |
| Partition pruning | การที่ Spark ข้าม partition ที่ไม่เกี่ยวข้อง และอ่านเฉพาะ partition ที่ match กับ filter โดยเฉพาะเมื่อ filter ตาม partition column |
| Over-partitioning | การเลือก partition column ที่ละเอียดหรือ cardinality สูงเกินไปจนเกิด folder/files เยอะเกินจำเป็น |
| DataFrame partition | การแบ่งข้อมูลเพื่อกระจายงานประมวลผลใน Spark runtime เช่น `repartition()` ไม่ใช่สิ่งเดียวกับ table partition จาก `partitionBy()` |

## Concept explanation

### Append คืออะไร?

`append` คือการเพิ่มข้อมูลใหม่เข้า table เดิม เหมาะกับ table ที่เก็บ event หรือ transaction ใหม่เรื่อย ๆ เช่น:

- daily transaction batch
- raw ingestion batch
- application event log
- audit/run log

ตัวอย่าง pattern:

```python
df_new_batch.write.format("delta").mode("append").saveAsTable("target_table")
```

แต่ข้อควรระวังคือ `append` ไม่ได้รู้เองว่า records ไหนเคยถูกเขียนไปแล้ว ถ้า rerun batch เดิมซ้ำโดยไม่มี business key / batch control / merge logic อาจเกิดข้อมูลซ้ำทันที

> Append is simple, but not automatically idempotent.

### Overwrite คืออะไร?

`overwrite` คือการเขียนข้อมูลใหม่แทนข้อมูลเดิม เหมาะกับกรณีเช่น:

- demo/test table ที่ต้อง reset ก่อน run ใหม่
- full snapshot table ที่ source ส่ง snapshot ล่าสุดมาทั้งชุด
- temporary/staging table ที่สร้างใหม่ได้เสมอ
- small lookup table ที่ตั้งใจ replace ทั้ง table

ตัวอย่าง pattern:

```python
df_snapshot.write.format("delta").mode("overwrite").saveAsTable("snapshot_table")
```

ข้อควรระวังคือ `overwrite` อาจแทนข้อมูลทั้ง table ถ้าใช้กับ fact/history table ผิดตัวอาจทำให้ข้อมูลเก่าหายได้

> Overwrite should be intentional, reviewed, and usually limited to safe targets.

### Partitioning คืออะไร?

`partitionBy` เป็นการบอก Spark ว่าตอน write table ให้จัด file layout แยกตาม column ที่กำหนด เช่น `sales_year` และ `sales_month`

ตัวอย่าง pattern:

```python
df.write.format("delta") \
    .mode("overwrite") \
    .partitionBy("sales_year", "sales_month") \
    .saveAsTable("partitioned_table")
```

Partitioning เหมาะเมื่อ query ส่วนใหญ่ filter ด้วย column นั้นจริง เช่น report ส่วนใหญ่ filter ตาม month หรือ date

แต่ไม่ควร partition ด้วย column ที่มีค่าหลากหลายมากเกินไป เช่น `transaction_id` เพราะจะทำให้มี partition จำนวนมากและเสี่ยง small files

### DataFrame partition vs table partition

คำว่า partition มี 2 ความหมายที่ต้องแยกให้ออก:

| Type | เกิดที่ไหน | ใช้ทำอะไร |
|---|---|---|
| DataFrame partition | Spark runtime | แบ่งข้อมูลเพื่อกระจายงานประมวลผลระหว่าง executors |
| Table partition | Lakehouse file layout | แบ่ง files/folders ตอน write table ตามค่าใน partition column |

ในวันนี้เราจะโฟกัสที่ **table partition** จาก `partitionBy()` ก่อน ส่วน `repartition()` และ `coalesce()` จะลงลึกอีกทีใน Day 25


## Mermaid diagram: Write Mode and Partitioning Flow

```mermaid
flowchart LR
    A[Batch 1 DataFrame] --> B[overwrite initial demo table]
    B --> C[(Lakehouse Table)]

    D[Batch 2 DataFrame] --> E[append new rows]
    E --> C

    F[Corrected Snapshot DataFrame] --> G[overwrite snapshot table]
    G --> H[(Snapshot Table)]

    I[Sales DataFrame with year/month] --> J[partitionBy sales_year, sales_month]
    J --> K[(Partitioned Lakehouse Table)]
    K --> L[Query filter by year/month]
```

Key idea:

> `append` เพิ่ม rows, `overwrite` แทนข้อมูลเดิม, ส่วน `partitionBy` เปลี่ยน layout ตอนเขียน table เพื่อช่วย query pattern บางแบบ

## Setup / imports

In [1]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

StatementMeta(, 855349cb-bb2d-43ba-b711-07868c1a8b5b, 3, Finished, Available, Finished, False)

## Create mock data

Dataset นี้จำลอง sales transactions จากหลายวันและหลายเดือน เพื่อใช้ทดสอบ:

- initial overwrite
- append new batch
- duplicate risk จาก append ซ้ำ
- full snapshot overwrite
- partitioned table by year/month

In [2]:
sales_schema = T.StructType([
    T.StructField("transaction_id", T.StringType(), False),
    T.StructField("customer_id", T.IntegerType(), False),
    T.StructField("region", T.StringType(), True),
    T.StructField("sales_date", T.StringType(), True),
    T.StructField("amount", T.DoubleType(), True),
    T.StructField("payment_status", T.StringType(), True),
    T.StructField("source_system", T.StringType(), True),
    T.StructField("load_batch_id", T.StringType(), True),
])

sales_batch_1_data = [
    ("TXN001", 101, "Bangkok", "2026-05-01", 1200.50, "success", "pos", "batch_001"),
    ("TXN002", 102, "Bangkok", "2026-05-01", 250.00, "success", "pos", "batch_001"),
    ("TXN003", 103, "Chiang Mai", "2026-05-02", 3400.00, "success", "ecommerce", "batch_001"),
    ("TXN004", 104, "Rayong", "2026-05-03", 0.00, "failed", "pos", "batch_001"),
    ("TXN005", 105, "Phuket", "2026-05-04", 780.25, "pending", "ecommerce", "batch_001"),
]

sales_batch_2_data = [
    ("TXN006", 106, "Bangkok", "2026-05-05", 5600.00, "success", "pos", "batch_002"),
    ("TXN007", 107, "Chiang Mai", "2026-05-06", 980.00, "success", "ecommerce", "batch_002"),
    ("TXN008", 108, "Rayong", "2026-06-01", 1500.00, "success", "pos", "batch_002"),
    ("TXN009", 109, "Bangkok", "2026-06-02", 220.00, "failed", "ecommerce", "batch_002"),
]


def prepare_sales_df(data):
    # Create a standardized sales DataFrame with date and partition columns.
    return (
        spark.createDataFrame(data, sales_schema)
        .withColumn("sales_date", F.to_date("sales_date"))
        .withColumn("sales_year", F.year("sales_date"))
        .withColumn("sales_month", F.month("sales_date"))
        .withColumn("ingestion_timestamp", F.current_timestamp())
    )


df_sales_batch_1 = prepare_sales_df(sales_batch_1_data)
df_sales_batch_2 = prepare_sales_df(sales_batch_2_data)

print("Batch 1 preview")
df_sales_batch_1.show(truncate=False)

print("Batch 2 preview")
df_sales_batch_2.show(truncate=False)

StatementMeta(, 855349cb-bb2d-43ba-b711-07868c1a8b5b, 4, Finished, Available, Finished, False)

Batch 1 preview
+--------------+-----------+----------+----------+------+--------------+-------------+-------------+----------+-----------+--------------------------+
|transaction_id|customer_id|region    |sales_date|amount|payment_status|source_system|load_batch_id|sales_year|sales_month|ingestion_timestamp       |
+--------------+-----------+----------+----------+------+--------------+-------------+-------------+----------+-----------+--------------------------+
|TXN001        |101        |Bangkok   |2026-05-01|1200.5|success       |pos          |batch_001    |2026      |5          |2026-06-02 04:54:21.703675|
|TXN002        |102        |Bangkok   |2026-05-01|250.0 |success       |pos          |batch_001    |2026      |5          |2026-06-02 04:54:21.703675|
|TXN003        |103        |Chiang Mai|2026-05-02|3400.0|success       |ecommerce    |batch_001    |2026      |5          |2026-06-02 04:54:21.703675|
|TXN004        |104        |Rayong    |2026-05-03|0.0   |failed        |pos   

## PySpark code examples

ใน section นี้จะไล่จากการสร้าง table ด้วย `overwrite` → เพิ่มข้อมูลด้วย `append` → เห็น duplicate risk → ใช้ `overwrite` กับ snapshot → เขียน partitioned table → inspect metadata ของ table

### Example 1: Create an initial table with overwrite

ในตัวอย่างแรก เราจะสร้าง demo table ด้วย `overwrite` เพื่อให้ Run all แล้วได้ผลลัพธ์ซ้ำได้ค่อนข้างปลอดภัย

แนวคิดคือใช้ `overwrite` เพื่อ reset เฉพาะ demo table ของวันนี้ ไม่ใช่ production table จริง

In [3]:
base_table_name = "day14_sales_write_modes"

(
    df_sales_batch_1.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(base_table_name)
)

print(f"Created table: {base_table_name}")
spark.read.table(base_table_name).orderBy("transaction_id").show(truncate=False)
print("Row count after initial overwrite:", spark.read.table(base_table_name).count())

StatementMeta(, 855349cb-bb2d-43ba-b711-07868c1a8b5b, 5, Finished, Available, Finished, False)

Created table: day14_sales_write_modes
+--------------+-----------+----------+----------+------+--------------+-------------+-------------+----------+-----------+-------------------------+
|transaction_id|customer_id|region    |sales_date|amount|payment_status|source_system|load_batch_id|sales_year|sales_month|ingestion_timestamp      |
+--------------+-----------+----------+----------+------+--------------+-------------+-------------+----------+-----------+-------------------------+
|TXN001        |101        |Bangkok   |2026-05-01|1200.5|success       |pos          |batch_001    |2026      |5          |2026-06-02 04:56:18.10335|
|TXN002        |102        |Bangkok   |2026-05-01|250.0 |success       |pos          |batch_001    |2026      |5          |2026-06-02 04:56:18.10335|
|TXN003        |103        |Chiang Mai|2026-05-02|3400.0|success       |ecommerce    |batch_001    |2026      |5          |2026-06-02 04:56:18.10335|
|TXN004        |104        |Rayong    |2026-05-03|0.0   |fail

### Example 2: Append a new batch

`append` จะเพิ่ม rows จาก batch ใหม่เข้า table เดิม โดยไม่ลบ batch เก่า

หลัง append แล้ว row count ควรเพิ่มจาก 5 เป็น 9 rows

In [4]:
(
    df_sales_batch_2.write
    .format("delta")
    .mode("append")
    .saveAsTable(base_table_name)
)

print(f"Appended batch 2 into table: {base_table_name}")

spark.read.table(base_table_name).orderBy("transaction_id").show(truncate=False)

spark.read.table(base_table_name).groupBy("load_batch_id").count().orderBy("load_batch_id").show()
print("Row count after append:", spark.read.table(base_table_name).count())

StatementMeta(, 855349cb-bb2d-43ba-b711-07868c1a8b5b, 6, Finished, Available, Finished, False)

Appended batch 2 into table: day14_sales_write_modes
+--------------+-----------+----------+----------+------+--------------+-------------+-------------+----------+-----------+-------------------------+
|transaction_id|customer_id|region    |sales_date|amount|payment_status|source_system|load_batch_id|sales_year|sales_month|ingestion_timestamp      |
+--------------+-----------+----------+----------+------+--------------+-------------+-------------+----------+-----------+-------------------------+
|TXN001        |101        |Bangkok   |2026-05-01|1200.5|success       |pos          |batch_001    |2026      |5          |2026-06-02 04:56:18.10335|
|TXN002        |102        |Bangkok   |2026-05-01|250.0 |success       |pos          |batch_001    |2026      |5          |2026-06-02 04:56:18.10335|
|TXN003        |103        |Chiang Mai|2026-05-02|3400.0|success       |ecommerce    |batch_001    |2026      |5          |2026-06-02 04:56:18.10335|
|TXN004        |104        |Rayong    |2026-05-

### Example 3: Append can create duplicates if the same batch is rerun

ตัวอย่างนี้ตั้งใจสร้าง duplicate demo table แยกออกมา เพื่อให้เห็นว่า `append` เฉย ๆ ไม่ได้กันการเขียนซ้ำ

ใน production ถ้า pipeline rerun batch เดิม ต้องมี logic เช่น batch control, deduplication, MERGE/UPSERT หรือ idempotent write pattern ไม่ใช่ append ทับไปเรื่อย ๆ

In [5]:
duplicate_demo_table_name = "day14_append_duplicate_demo"

(
    df_sales_batch_1.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(duplicate_demo_table_name)
)

# Simulate rerunning the same batch with append.
(
    df_sales_batch_1.write
    .format("delta")
    .mode("append")
    .saveAsTable(duplicate_demo_table_name)
)

print(f"Duplicate demo table: {duplicate_demo_table_name}")

spark.read.table(duplicate_demo_table_name).groupBy("transaction_id").count().orderBy("transaction_id").show()

spark.read.table(duplicate_demo_table_name).filter(F.col("transaction_id") == "TXN001").show(truncate=False)

StatementMeta(, 855349cb-bb2d-43ba-b711-07868c1a8b5b, 7, Finished, Available, Finished, False)

Duplicate demo table: day14_append_duplicate_demo
+--------------+-----+
|transaction_id|count|
+--------------+-----+
|        TXN001|    2|
|        TXN002|    2|
|        TXN003|    2|
|        TXN004|    2|
|        TXN005|    2|
+--------------+-----+

+--------------+-----------+-------+----------+------+--------------+-------------+-------------+----------+-----------+--------------------------+
|transaction_id|customer_id|region |sales_date|amount|payment_status|source_system|load_batch_id|sales_year|sales_month|ingestion_timestamp       |
+--------------+-----------+-------+----------+------+--------------+-------------+-------------+----------+-----------+--------------------------+
|TXN001        |101        |Bangkok|2026-05-01|1200.5|success       |pos          |batch_001    |2026      |5          |2026-06-02 04:58:35.708785|
|TXN001        |101        |Bangkok|2026-05-01|1200.5|success       |pos          |batch_001    |2026      |5          |2026-06-02 04:58:31.285565|
+-

### Example 4: Overwrite is useful for full snapshot tables

บาง source ไม่ได้ส่ง incremental batch แต่ส่ง full snapshot ล่าสุดมาทั้งชุด เช่น lookup/master table ขนาดเล็ก

ในกรณีนี้ `overwrite` อาจเหมาะกว่า `append` เพราะเราไม่ได้อยากเก็บ snapshot เก่าซ้ำใน table เดิม

In [6]:
snapshot_table_name = "day14_sales_snapshot"

snapshot_v1_data = [
    ("TXN101", 201, "Bangkok", "2026-05-01", 1000.00, "success", "snapshot_api", "snapshot_v1"),
    ("TXN102", 202, "Chiang Mai", "2026-05-01", 2000.00, "success", "snapshot_api", "snapshot_v1"),
    ("TXN103", 203, "Phuket", "2026-05-01", 3000.00, "pending", "snapshot_api", "snapshot_v1"),
]

snapshot_v2_data = [
    ("TXN101", 201, "Bangkok", "2026-05-01", 1000.00, "success", "snapshot_api", "snapshot_v2"),
    ("TXN102", 202, "Chiang Mai", "2026-05-01", 2100.00, "success", "snapshot_api", "snapshot_v2"),
    ("TXN104", 204, "Rayong", "2026-05-02", 4500.00, "success", "snapshot_api", "snapshot_v2"),
]

df_snapshot_v1 = prepare_sales_df(snapshot_v1_data)
df_snapshot_v2 = prepare_sales_df(snapshot_v2_data)

(
    df_snapshot_v1.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(snapshot_table_name)
)

print("Snapshot v1")
spark.read.table(snapshot_table_name).orderBy("transaction_id").show(truncate=False)

(
    df_snapshot_v2.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(snapshot_table_name)
)

print("Snapshot v2 after overwrite")
spark.read.table(snapshot_table_name).orderBy("transaction_id").show(truncate=False)
print("Row count after snapshot overwrite:", spark.read.table(snapshot_table_name).count())

StatementMeta(, 855349cb-bb2d-43ba-b711-07868c1a8b5b, 8, Finished, Available, Finished, False)

Snapshot v1
+--------------+-----------+----------+----------+------+--------------+-------------+-------------+----------+-----------+--------------------------+
|transaction_id|customer_id|region    |sales_date|amount|payment_status|source_system|load_batch_id|sales_year|sales_month|ingestion_timestamp       |
+--------------+-----------+----------+----------+------+--------------+-------------+-------------+----------+-----------+--------------------------+
|TXN101        |201        |Bangkok   |2026-05-01|1000.0|success       |snapshot_api |snapshot_v1  |2026      |5          |2026-06-02 05:01:42.605865|
|TXN102        |202        |Chiang Mai|2026-05-01|2000.0|success       |snapshot_api |snapshot_v1  |2026      |5          |2026-06-02 05:01:42.605865|
|TXN103        |203        |Phuket    |2026-05-01|3000.0|pending       |snapshot_api |snapshot_v1  |2026      |5          |2026-06-02 05:01:42.605865|
+--------------+-----------+----------+----------+------+--------------+----------

### Example 5: Write a partitioned Lakehouse table

ตอนนี้เราจะรวม batch 1 และ batch 2 แล้ว write เป็น partitioned table โดยใช้ `sales_year` และ `sales_month`

เหตุผลที่เลือก year/month เพราะ query/report มัก filter ตามช่วงเวลา และ cardinality ไม่สูงเกินไปสำหรับ demo นี้

In [7]:
partitioned_table_name = "day14_sales_partitioned"

df_all_sales = df_sales_batch_1.unionByName(df_sales_batch_2)

(
    df_all_sales.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("sales_year", "sales_month")
    .saveAsTable(partitioned_table_name)
)

print(f"Created partitioned table: {partitioned_table_name}")

spark.read.table(partitioned_table_name).groupBy("sales_year", "sales_month").count().orderBy("sales_year", "sales_month").show()
spark.read.table(partitioned_table_name).orderBy("sales_date", "transaction_id").show(truncate=False)

StatementMeta(, 855349cb-bb2d-43ba-b711-07868c1a8b5b, 9, Finished, Available, Finished, False)

Created partitioned table: day14_sales_partitioned
+----------+-----------+-----+
|sales_year|sales_month|count|
+----------+-----------+-----+
|      2026|          5|    7|
|      2026|          6|    2|
+----------+-----------+-----+

+--------------+-----------+----------+----------+------+--------------+-------------+-------------+----------+-----------+--------------------------+
|transaction_id|customer_id|region    |sales_date|amount|payment_status|source_system|load_batch_id|sales_year|sales_month|ingestion_timestamp       |
+--------------+-----------+----------+----------+------+--------------+-------------+-------------+----------+-----------+--------------------------+
|TXN001        |101        |Bangkok   |2026-05-01|1200.5|success       |pos          |batch_001    |2026      |5          |2026-06-02 05:04:17.842964|
|TXN002        |102        |Bangkok   |2026-05-01|250.0 |success       |pos          |batch_001    |2026      |5          |2026-06-02 05:04:17.842964|
|TXN003

### Example 6: Inspect table metadata with DESCRIBE DETAIL

`DESCRIBE DETAIL` ช่วยให้เห็น metadata ระดับ table เช่น format, location, partition columns, จำนวน files และขนาดข้อมูลของ table

ใน Fabric Lakehouse จุดนี้ช่วยให้เข้าใจมากขึ้นว่า Lakehouse table ไม่ได้เป็นแค่ชื่อ table เฉย ๆ แต่มี files และ folder layout อยู่ด้านหลังด้วย

In [8]:
partition_table_detail_df = spark.sql(f"DESCRIBE DETAIL {partitioned_table_name}")

partition_table_detail_df.select(
    "format",
    "location",
    "partitionColumns",
    "numFiles"
).show(truncate=False)

partition_table_detail = partition_table_detail_df.select(
    "location",
    "partitionColumns",
    "numFiles"
).first()  # first() returns the first metadata row as a Python Row object.

partition_table_location = partition_table_detail["location"]  # Extract the table location value from the Row object.
partition_columns = partition_table_detail["partitionColumns"]  # Extract partition column list from the Row object.
partition_num_files = partition_table_detail["numFiles"]  # Extract current file count from the Row object.

print("Table location:", partition_table_location)
print("Partition columns:", partition_columns)
print("Number of files:", partition_num_files)

StatementMeta(, 855349cb-bb2d-43ba-b711-07868c1a8b5b, 10, Finished, Available, Finished, False)

+------+-------------------------------------------------------------------------------------------------------------------------------------------------+-------------------------+--------+
|format|location                                                                                                                                         |partitionColumns         |numFiles|
+------+-------------------------------------------------------------------------------------------------------------------------------------------------+-------------------------+--------+
|delta |abfss://a5de4c5d-a2ed-4a31-923d-ff4fcd64006e@onelake.dfs.fabric.microsoft.com/8f868c6a-7fc0-4ade-a075-39cee1694ad9/Tables/day14_sales_partitioned|[sales_year, sales_month]|2       |
+------+-------------------------------------------------------------------------------------------------------------------------------------------------+-------------------------+--------+

Table location: abfss://a5de4c5d-a2ed-4a31-923d-f

### Example 7: Filter by partition columns

ถ้า table ถูก partition ด้วย `sales_year` และ `sales_month` การ query ที่ filter ด้วย columns เหล่านี้มีโอกาสได้ประโยชน์จาก **partition pruning** เพราะ Spark สามารถใช้ filter ระดับ partition เพื่อข้าม partition ที่ไม่เกี่ยวข้องได้

ใน output ของ `.explain(True)` ให้สังเกตที่ Physical Plan ถ้าเห็น `PartitionFilters` บน `sales_year` และ `sales_month` แปลว่า Spark recognize filter นี้เป็น partition-level filter แล้ว

In [9]:
df_may_2026_sales = (
    spark.read.table(partitioned_table_name)
    .filter((F.col("sales_year") == 2026) & (F.col("sales_month") == 5))
)

print("May 2026 sales")
df_may_2026_sales.orderBy("sales_date", "transaction_id").show(truncate=False)

print("Explain plan for partition-filtered query")
df_may_2026_sales.explain(True)

StatementMeta(, 855349cb-bb2d-43ba-b711-07868c1a8b5b, 11, Finished, Available, Finished, False)

May 2026 sales
+--------------+-----------+----------+----------+------+--------------+-------------+-------------+----------+-----------+--------------------------+
|transaction_id|customer_id|region    |sales_date|amount|payment_status|source_system|load_batch_id|sales_year|sales_month|ingestion_timestamp       |
+--------------+-----------+----------+----------+------+--------------+-------------+-------------+----------+-----------+--------------------------+
|TXN001        |101        |Bangkok   |2026-05-01|1200.5|success       |pos          |batch_001    |2026      |5          |2026-06-02 05:04:17.842964|
|TXN002        |102        |Bangkok   |2026-05-01|250.0 |success       |pos          |batch_001    |2026      |5          |2026-06-02 05:04:17.842964|
|TXN003        |103        |Chiang Mai|2026-05-02|3400.0|success       |ecommerce    |batch_001    |2026      |5          |2026-06-02 05:04:17.842964|
|TXN004        |104        |Rayong    |2026-05-03|0.0   |failed        |pos    

### Example 8: Bad partition choice example

ตัวอย่างนี้ไม่ write table จริง แต่ช่วยให้เห็นว่า partition column ที่ cardinality สูงเกินไป เช่น `transaction_id` มักไม่เหมาะ

ถ้า partition ตาม `transaction_id` อาจได้ partition จำนวนใกล้เคียงจำนวน rows ซึ่งทำให้ file layout กระจัดกระจายและดูแลยาก

In [ ]:
partition_candidate_summary = (
    df_all_sales
    .select(
        F.count("*").alias("row_count"),
        F.countDistinct("sales_year").alias("distinct_sales_year"),
        F.countDistinct("sales_month").alias("distinct_sales_month"),
        F.countDistinct("sales_date").alias("distinct_sales_date"),
        F.countDistinct("transaction_id").alias("distinct_transaction_id"),
        F.countDistinct("customer_id").alias("distinct_customer_id"),
    )
)

partition_candidate_summary.show(truncate=False)

print("Rule of thumb: avoid partition columns that are almost as unique as row-level identifiers.")

## Quick recap

| Question | Answer |
|---|---|
| `append` ทำอะไร? | เพิ่ม rows ใหม่เข้า table เดิมโดยไม่ลบของเก่า |
| `overwrite` ทำอะไร? | เขียนข้อมูลใหม่แทนข้อมูลเดิมของ table/path นั้น |
| ใช้ `append` แล้ว rerun batch เดิมปลอดภัยไหม? | ไม่เสมอไป อาจเกิด duplicate ถ้าไม่มี idempotency หรือ merge logic |
| `partitionBy` ใช้ทำอะไร? | จัด file layout ของ table ตาม partition columns ตอน write |
| ควร partition ด้วย column แบบไหน? | Column ที่ใช้ filter บ่อยและ cardinality ไม่สูงเกินไป เช่น year/month/date |
| DataFrame partition กับ table partition เหมือนกันไหม? | ไม่เหมือนกัน DataFrame partition คือ runtime execution ส่วน table partition คือ file layout |

## Exercises

### Exercise 1: Append one more daily batch

เพิ่ม batch ที่ 3 เข้า `day14_sales_write_modes`

Requirements:

- สร้าง DataFrame ใหม่ชื่อ `df_sales_batch_3`
- ใช้ schema และ helper function เดิม
- append เข้า `base_table_name`
- ตรวจ row count และจำนวน rows ต่อ `load_batch_id`

Expected result:

- table มี rows เพิ่มขึ้นจาก batch 3
- เห็น `batch_003` ใน summary

In [10]:
sales_batch_3_data = [
    ("TXN010", 110, "Bangkok", "2026-06-03", 890.00, "success", "pos", "batch_003"),
    ("TXN011", 111, "Phuket", "2026-06-03", 1250.00, "success", "ecommerce", "batch_003"),
    ("TXN012", 112, "Chiang Mai", "2026-06-04", 300.00, "pending", "pos", "batch_003"),
]

df_sales_batch_3 = prepare_sales_df(sales_batch_3_data)

before_append_count = spark.read.table(base_table_name).count()

(
    df_sales_batch_3.write
    .format("delta")
    .mode("append")
    .saveAsTable(base_table_name)
)

after_append_count = spark.read.table(base_table_name).count()

print("Before append count:", before_append_count)
print("After append count:", after_append_count)
print("Rows added:", after_append_count - before_append_count)

spark.read.table(base_table_name).groupBy("load_batch_id").count().orderBy("load_batch_id").show()

StatementMeta(, 855349cb-bb2d-43ba-b711-07868c1a8b5b, 12, Finished, Available, Finished, False)

Before append count: 9
After append count: 12
Rows added: 3
+-------------+-----+
|load_batch_id|count|
+-------------+-----+
|    batch_001|    5|
|    batch_002|    4|
|    batch_003|    3|
+-------------+-----+



### Exercise 2: Overwrite a small snapshot table safely

สร้าง snapshot table แยกสำหรับ exercise แล้ว overwrite ด้วย snapshot version ใหม่

Requirements:

- ใช้ table name `day14_exercise_snapshot`
- เขียน snapshot v1 ด้วย `overwrite`
- เขียน snapshot v2 ด้วย `overwrite`
- ตรวจว่า row ที่หายจาก snapshot v2 ไม่เหลืออยู่ใน table แล้ว

Expected result:

- หลัง overwrite ด้วย v2 จะเห็นเฉพาะ records ของ snapshot ล่าสุด

In [11]:
exercise_snapshot_table_name = "day14_exercise_snapshot"

exercise_snapshot_v1_data = [
    ("TXN201", 301, "Bangkok", "2026-05-10", 700.00, "success", "snapshot_api", "exercise_v1"),
    ("TXN202", 302, "Rayong", "2026-05-10", 900.00, "success", "snapshot_api", "exercise_v1"),
]

exercise_snapshot_v2_data = [
    ("TXN201", 301, "Bangkok", "2026-05-10", 750.00, "success", "snapshot_api", "exercise_v2"),
    ("TXN203", 303, "Chiang Mai", "2026-05-11", 1100.00, "success", "snapshot_api", "exercise_v2"),
]

df_exercise_snapshot_v1 = prepare_sales_df(exercise_snapshot_v1_data)
df_exercise_snapshot_v2 = prepare_sales_df(exercise_snapshot_v2_data)

(
    df_exercise_snapshot_v1.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(exercise_snapshot_table_name)
)

print("Exercise snapshot v1")
spark.read.table(exercise_snapshot_table_name).orderBy("transaction_id").show(truncate=False)

(
    df_exercise_snapshot_v2.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(exercise_snapshot_table_name)
)

print("Exercise snapshot v2 after overwrite")
spark.read.table(exercise_snapshot_table_name).orderBy("transaction_id").show(truncate=False)

print("Check removed transaction TXN202")
spark.read.table(exercise_snapshot_table_name).filter(F.col("transaction_id") == "TXN202").show(truncate=False)

StatementMeta(, 855349cb-bb2d-43ba-b711-07868c1a8b5b, 13, Finished, Available, Finished, False)

Exercise snapshot v1
+--------------+-----------+-------+----------+------+--------------+-------------+-------------+----------+-----------+-------------------------+
|transaction_id|customer_id|region |sales_date|amount|payment_status|source_system|load_batch_id|sales_year|sales_month|ingestion_timestamp      |
+--------------+-----------+-------+----------+------+--------------+-------------+-------------+----------+-----------+-------------------------+
|TXN201        |301        |Bangkok|2026-05-10|700.0 |success       |snapshot_api |exercise_v1  |2026      |5          |2026-06-02 05:21:32.51314|
|TXN202        |302        |Rayong |2026-05-10|900.0 |success       |snapshot_api |exercise_v1  |2026      |5          |2026-06-02 05:21:32.51314|
+--------------+-----------+-------+----------+------+--------------+-------------+-------------+----------+-----------+-------------------------+

Exercise snapshot v2 after overwrite
+--------------+-----------+----------+----------+------+--

### Exercise 3: Create a partitioned monthly table

สร้าง table ใหม่ที่ partition ด้วย `sales_year` และ `sales_month`

Requirements:

- ใช้ table name `day14_exercise_partitioned_sales`
- รวม batch 1, 2 และ 3 ด้วย `unionByName`
- เขียนด้วย `partitionBy("sales_year", "sales_month")`
- ใช้ `DESCRIBE DETAIL` เพื่อตรวจ partition columns
- filter เฉพาะ June 2026 แล้วแสดงผล

Expected result:

- `partitionColumns` ควรเป็น `sales_year`, `sales_month`
- June 2026 ควรมี records จาก batch 2 และ batch 3

In [12]:
exercise_partitioned_table_name = "day14_exercise_partitioned_sales"

df_exercise_all_sales = (
    df_sales_batch_1
    .unionByName(df_sales_batch_2)
    .unionByName(df_sales_batch_3)
)

(
    df_exercise_all_sales.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("sales_year", "sales_month")
    .saveAsTable(exercise_partitioned_table_name)
)

spark.sql(f"DESCRIBE DETAIL {exercise_partitioned_table_name}").select(
    "format",
    "partitionColumns",
    "numFiles"
).show(truncate=False)

print("June 2026 records")
(
    spark.read.table(exercise_partitioned_table_name)
    .filter((F.col("sales_year") == 2026) & (F.col("sales_month") == 6))
    .orderBy("sales_date", "transaction_id")
    .show(truncate=False)
)

StatementMeta(, 855349cb-bb2d-43ba-b711-07868c1a8b5b, 14, Finished, Available, Finished, False)

+------+-------------------------+--------+
|format|partitionColumns         |numFiles|
+------+-------------------------+--------+
|delta |[sales_year, sales_month]|2       |
+------+-------------------------+--------+

June 2026 records
+--------------+-----------+----------+----------+------+--------------+-------------+-------------+----------+-----------+--------------------------+
|transaction_id|customer_id|region    |sales_date|amount|payment_status|source_system|load_batch_id|sales_year|sales_month|ingestion_timestamp       |
+--------------+-----------+----------+----------+------+--------------+-------------+-------------+----------+-----------+--------------------------+
|TXN008        |108        |Rayong    |2026-06-01|1500.0|success       |pos          |batch_002    |2026      |6          |2026-06-02 05:23:46.016521|
|TXN009        |109        |Bangkok   |2026-06-02|220.0 |failed        |ecommerce    |batch_002    |2026      |6          |2026-06-02 05:23:46.016521|
|TXN01

## Common mistakes

1. **ใช้ `overwrite` กับ production fact table โดยไม่ตั้งใจ**

   `overwrite` อาจแทนข้อมูลเดิมทั้ง table ถ้าใช้ผิด table อาจทำให้ historical data หายได้ ควรใช้กับ demo/staging/snapshot target ที่ชัดเจนเท่านั้น

2. **คิดว่า `append` ป้องกันข้อมูลซ้ำเอง**

   `append` แค่เพิ่ม rows ไม่ได้ตรวจ business key หรือ batch ที่เคยเขียนไปแล้ว ถ้า rerun batch เดิมซ้ำก็ duplicate ได้

3. **ใช้ `append` กับ full snapshot source**

   ถ้า source ส่ง snapshot ล่าสุดมาทั้งชุด การ append snapshot ทุกวันอาจทำให้ business key เดิม เช่น `customer_id` ถูกเก็บซ้ำหลายรอบ ควรแยกให้ชัดว่า table นี้ต้องการเก็บ current snapshot หรือ historical snapshot

4. **Partition ด้วย column ที่ cardinality สูงเกินไป**

   เช่น `transaction_id`, `customer_id` หรือ timestamp ระดับวินาที อาจสร้าง partition/files มากเกินไปและนำไปสู่ small files problem

5. **Partition ด้วย column ที่ query ไม่ค่อยใช้ filter**

   ถ้า query ส่วนใหญ่ไม่ได้ filter ตาม partition column ก็แทบไม่ได้ประโยชน์จาก partition pruning และอาจเพิ่มภาระจาก folders/partitions และ small files ที่ต้องดูแลมากขึ้น

6. **สับสนระหว่าง DataFrame partition กับ table partition**

   `repartition()` เกี่ยวกับ Spark execution runtime ส่วน `partitionBy()` ตอน write เกี่ยวกับ file layout ของ table

7. **ใช้ `overwriteSchema` แบบไม่ตั้งใจ**

   `overwriteSchema` มีประโยชน์เวลา reset demo table หรือเปลี่ยน schema แบบตั้งใจ แต่ไม่ควรใช้เพื่อกลบ schema problem ใน production pipeline


## Expected Output / Success Criteria

เมื่อจบ Day 14 ควรทำได้:

- อธิบายได้ว่า `append` คือการเพิ่ม rows ใหม่เข้า table เดิม
- อธิบายได้ว่า `overwrite` คือการเขียนข้อมูลใหม่แทนข้อมูลเดิม และต้องใช้อย่างระวัง
- แยก use case ระหว่าง incremental/event table กับ full snapshot table ได้
- เขียน Lakehouse table ด้วย `.mode("append")` และ `.mode("overwrite")` ได้
- เข้าใจว่า `append` ไม่ได้ทำให้ pipeline idempotent โดยอัตโนมัติ
- เขียน partitioned table ด้วย `.partitionBy("sales_year", "sales_month")` ได้
- ใช้ `DESCRIBE DETAIL` เพื่อตรวจ table metadata เช่น format, location, partition columns และ file count ได้
- อธิบายได้ว่า partition column ควรเลือกจาก query/filter pattern และ cardinality ที่เหมาะสม
- อธิบายได้ว่า DataFrame partition กับ table partition ไม่ใช่เรื่องเดียวกัน
- ระวัง accidental overwrite, duplicate append, over-partitioning และ `overwriteSchema` ที่ใช้ผิดบริบทได้

## Final takeaway

Append, overwrite และ partitioning เป็นพื้นฐานของ Lakehouse table operation ที่ดูเหมือนง่าย แต่มีผลต่อความถูกต้องและ performance ของ pipeline โดยตรง

> ก่อน write table ทุกครั้ง ให้ตอบให้ได้ว่าเรากำลังเพิ่มข้อมูลใหม่ แทน snapshot เดิม หรือเปลี่ยน file layout เพื่อ query pattern อะไร

Mindset ที่ควรจำ:

- `append` เหมาะกับ new records แต่ต้องออกแบบกัน duplicate/rerun เอง
- `overwrite` เหมาะกับ reset/snapshot/staging แต่ต้องระวัง target table มาก
- `partitionBy` ควรเลือกจาก column ที่ filter บ่อยและ cardinality ไม่สูงเกินไป
- write mode เป็น data correctness decision ไม่ใช่แค่ syntax
- partitioning เป็น file layout decision และจะช่วย performance ได้ก็ต่อเมื่อเลือก partition column สอดคล้องกับ query pattern จริง

## Optional cleanup

In [ ]:
# spark.sql("DROP TABLE IF EXISTS day14_sales_write_modes")
# spark.sql("DROP TABLE IF EXISTS day14_append_duplicate_demo")
# spark.sql("DROP TABLE IF EXISTS day14_sales_snapshot")
# spark.sql("DROP TABLE IF EXISTS day14_sales_partitioned")
# spark.sql("DROP TABLE IF EXISTS day14_exercise_snapshot")
# spark.sql("DROP TABLE IF EXISTS day14_exercise_partitioned_sales")